<a href="https://colab.research.google.com/github/NataliaFarieta/nataliafarieta/blob/main/Cargue.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller ETL – Cubo SECOP
## Notebook 4 de 4: `Cargue.ipynb`
### Carga a la capa ORO, integración con Hive y consultas del cubo

**Universidad Central — Maestría en Analítica de Datos**
**Actividad:** T2 – Taller ETL – Cubo SECOP

---

Este es el notebook final del pipeline. Lee la capa **PLATA** (producida por
`Transformacion.ipynb`), la persiste en formato Parquet en la capa **ORO**
del *data lake*, carga esos datos en las tablas Hive creadas por
`CuboDatos.ipynb`, valida la carga, y ejecuta el conjunto de consultas
analíticas que demuestran que el cubo de datos funciona.


## 1. Introducción

La capa de cargue cierra el ciclo ETL: convierte el modelo dimensional ya
limpio y validado (PLATA) en un activo consultable de forma eficiente
(ORO + Hive). Aquí se aplican dos decisiones técnicas relevantes:

- **Formato Parquet** para la capa ORO, por ser un formato columnar
  optimizado para consultas analíticas — permite leer solo las columnas
  necesarias por consulta en lugar de filas completas (Chambers y Zaharia,
  2018).
- **Particionamiento por año de firma** en la tabla de hechos, justificado
  porque buena parte de las preguntas de negocio de este taller son de
  naturaleza temporal (evolución anual del valor contratado); particionar
  por una columna que **no** se usa en los filtros de las consultas no
  aportaría beneficio y sí añadiría complejidad innecesaria, por lo que
  las nueve dimensiones se cargan **sin particionar**.

Una vez cargados los datos en Hive, se ejecutan diez consultas SQL sobre
el cubo, cada una asociada explícitamente a una pregunta de negocio, según
el patrón *pregunta → datos necesarios → consulta → resultado →
interpretación → utilidad*.


## 2. Objetivos

### Objetivo general
Cargar el modelo dimensional de la capa PLATA en la capa ORO (Parquet) y en
las tablas Hive del cubo, y demostrar su utilidad analítica mediante un
conjunto representativo de consultas SQL.

### Objetivos específicos
1. Leer la capa PLATA producida por `Transformacion.ipynb`.
2. Persistir la tabla de hechos particionada por año de firma, y las
   dimensiones sin particionar, en la capa ORO (Parquet).
3. Cargar esos datos en las tablas Hive creadas por `CuboDatos.ipynb`.
4. Validar la carga mediante conteos y `DESCRIBE` de las tablas finales.
5. Ejecutar diez consultas analíticas sobre el cubo, cada una vinculada a
   una pregunta de negocio concreta, con su interpretación.
6. Documentar una guía de defensa del taller con las decisiones técnicas
   tomadas a lo largo del pipeline.


## 3. Configuración de la sesión de Spark


In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName("Cargue_SECOP") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.catalogImplementation", "hive") \
    .enableHiveSupport() \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

DB_NAME = "secop_cubo"
spark.catalog.setCurrentDatabase(DB_NAME)

print("SparkSession creada. Version de Spark:", spark.version)
print(f"Base de datos activa: {spark.catalog.currentDatabase()}")


## 4. Lectura de la capa PLATA


In [ ]:
RUTA_PLATA = "hdfs://namenode:9000/datalake/plata"
RUTA_ORO   = "hdfs://namenode:9000/datalake/oro"

nombres_tablas = [
    "hecho_contratos", "dim_geografia", "dim_sector", "dim_proveedor",
    "dim_entidad", "dim_modalidad_contratacion", "dim_tipo_contrato",
    "dim_producto", "dim_ordenador_gasto", "dim_supervisor",
]

tablas_plata = {}
lectura_exitosa = True

for nombre in nombres_tablas:
    ruta = f"{RUTA_PLATA}/{nombre}"
    try:
        tablas_plata[nombre] = spark.read.parquet(ruta)
        print(f"OK   -> {nombre:30s} ({tablas_plata[nombre].count():6d} filas) leido de {ruta}")
    except Exception as e:
        print(f"ERROR -> {nombre}: {e}")
        lectura_exitosa = False

if not lectura_exitosa:
    print("\nPENDIENTE DE EJECUCION Y VALIDACION: no se pudo leer alguna tabla de PLATA. "
          "Ejecutar primero Extraccion.ipynb y Transformacion.ipynb exitosamente "
          "en el entorno Docker del curso, en ese orden.")


## 5. Persistencia en la capa ORO (Parquet)

La tabla de hechos se particiona por `anio_firma` (columna derivada de
`fecha_de_firma`). Esta decisión se justifica porque el particionamiento
en Hive/Spark solo aporta valor cuando las consultas filtran habitualmente
por esa columna (poda de particiones o *partition pruning*); dado que
varias de las diez consultas de la sección 8 filtran o agrupan por año, el
particionamiento por año reduce el volumen de datos leído en esas
consultas. Las dimensiones, al ser tablas pequeñas de baja cardinalidad,
no se particionan: hacerlo generaría archivos pequeños sin beneficio real
de poda.


In [ ]:
if "hecho_contratos" in tablas_plata:
    hecho_oro = tablas_plata["hecho_contratos"].withColumn(
        "anio_firma", F.year(F.col("fecha_de_firma"))
    )

    hecho_oro.write \
        .mode("overwrite") \
        .partitionBy("anio_firma") \
        .parquet(f"{RUTA_ORO}/hecho_contratos")

    print(f"hecho_contratos escrito en ORO, particionado por anio_firma "
          f"({hecho_oro.count()} filas).")
else:
    print("PENDIENTE DE EJECUCION Y VALIDACION: no hay hecho_contratos en PLATA.")


In [ ]:
dimensiones_para_oro = {k: v for k, v in tablas_plata.items() if k != "hecho_contratos"}

for nombre, df_dim in dimensiones_para_oro.items():
    ruta = f"{RUTA_ORO}/{nombre}"
    df_dim.write.mode("overwrite").parquet(ruta)
    print(f"{nombre:30s} escrito en ORO ({df_dim.count()} filas), sin particionar.")


## 6. Cargue en las tablas Hive del cubo

Se cargan los datos de ORO en las tablas Hive creadas por
`CuboDatos.ipynb`, usando `INSERT OVERWRITE` a través de la API de
DataFrame (`saveAsTable` con modo `overwrite`), de forma que el notebook
sea reejecutable sin duplicar datos en corridas sucesivas.


In [ ]:
if "hecho_contratos" in tablas_plata:
    (
        hecho_oro
        .drop("anio_firma")  # la particion fisica en ORO no forma parte del esquema Hive definido en CuboDatos
        .write
        .mode("overwrite")
        .format("parquet")
        .saveAsTable(f"{DB_NAME}.hecho_contratos")
    )
    print("hecho_contratos cargada en Hive.")

for nombre, df_dim in dimensiones_para_oro.items():
    df_dim.write.mode("overwrite").format("parquet").saveAsTable(f"{DB_NAME}.{nombre}")
    print(f"{nombre} cargada en Hive.")


## 7. Validación de la carga


In [ ]:
spark.sql(f"SHOW TABLES IN {DB_NAME}").show(n=20, truncate=False)


In [ ]:
for nombre in nombres_tablas:
    try:
        total = spark.sql(f"SELECT COUNT(*) AS total FROM {DB_NAME}.{nombre}").collect()[0]["total"]
        print(f"{nombre:30s} -> {total:8d} filas en Hive")
    except Exception as e:
        print(f"{nombre:30s} -> ERROR al consultar: {e}")


In [ ]:
print("Esquema final de hecho_contratos en Hive:")
spark.sql(f"DESCRIBE {DB_NAME}.hecho_contratos").show(n=60, truncate=False)


## 8. Consultas del cubo

Cada consulta se documenta con el patrón: **pregunta de negocio → datos
necesarios → consulta SQL → resultado → interpretación → utilidad para la
decisión**. Las diez consultas cubren: conteo del hecho, una agregación de
medida real, las tres primeras dimensiones del modelo (sector, modalidad,
geografía), un `JOIN` simple, un `JOIN` con al menos tres dimensiones,
validación de huérfanos, análisis temporal y una pregunta analítica
adicional.


### Consulta 1 — Conteo de registros de la tabla de hechos

**Pregunta:** ¿Cuántos contratos quedaron finalmente cargados en el cubo?

**Datos necesarios:** `hecho_contratos`.

**Utilidad:** es la métrica base de control de todo el pipeline; cualquier
otra consulta de agregación debe ser consistente con este total.


In [ ]:
resultado_1 = spark.sql(f"""
SELECT COUNT(*) AS total_contratos
FROM {DB_NAME}.hecho_contratos
""")
resultado_1.show()


### Consulta 2 — Agregación de una medida real: valor total contratado

**Pregunta:** ¿Cuál es el valor total y el valor promedio de los contratos
cargados en el cubo?

**Datos necesarios:** `hecho_contratos.valor_del_contrato`.

**Interpretación esperada:** el valor total da una magnitud agregada del
gasto público capturado en la muestra; el promedio ayuda a contextualizar
si el conjunto está dominado por contratos de bajo valor (más frecuentes,
p. ej. mínima cuantía) o por unos pocos contratos grandes.

**Utilidad:** insumo directo para reportes ejecutivos de seguimiento de
contratación pública.


In [ ]:
resultado_2 = spark.sql(f"""
SELECT
    ROUND(SUM(valor_del_contrato), 2) AS valor_total_contratado,
    ROUND(AVG(valor_del_contrato), 2) AS valor_promedio_contrato,
    COUNT(*) AS contratos_considerados
FROM {DB_NAME}.hecho_contratos
WHERE valor_del_contrato IS NOT NULL
""")
resultado_2.show(truncate=False)


### Consulta 3 — Análisis con la primera dimensión real: Sector

**Pregunta:** ¿Qué sectores concentran mayor valor contratado?

**Datos necesarios:** `hecho_contratos` + `dim_sector`.

**Utilidad:** permite priorizar auditoría o seguimiento sobre los sectores
de mayor exposición presupuestal.


In [ ]:
resultado_3 = spark.sql(f"""
SELECT
    s.sector,
    COUNT(*) AS numero_contratos,
    ROUND(SUM(h.valor_del_contrato), 2) AS valor_total
FROM {DB_NAME}.hecho_contratos h
LEFT JOIN {DB_NAME}.dim_sector s
    ON h.sector_id_sector = s.id_sector
GROUP BY s.sector
ORDER BY valor_total DESC
""")
resultado_3.show(n=15, truncate=False)


### Consulta 4 — Análisis con la segunda dimensión real: Modalidad de contratación

**Pregunta:** ¿Qué modalidad de contratación es más frecuente y cuál
concentra más valor?

**Datos necesarios:** `hecho_contratos` + `dim_modalidad_contratacion`.

**Utilidad:** relevante para análisis de transparencia — comparar
frecuencia vs. valor ayuda a detectar si modalidades con menor
competencia (p. ej. contratación directa) concentran una proporción
desproporcionada del gasto.


In [ ]:
resultado_4 = spark.sql(f"""
SELECT
    m.modalidad_de_contratacion,
    COUNT(*) AS numero_contratos,
    ROUND(SUM(h.valor_del_contrato), 2) AS valor_total
FROM {DB_NAME}.hecho_contratos h
LEFT JOIN {DB_NAME}.dim_modalidad_contratacion m
    ON h.id_modalidad_de_contratacion = m.id_modalidad_de_contratacion
GROUP BY m.modalidad_de_contratacion
ORDER BY numero_contratos DESC
""")
resultado_4.show(n=15, truncate=False)


### Consulta 5 — Análisis con la tercera dimensión real: Geografía

**Pregunta:** ¿Qué departamentos concentran más contratos y más valor
contratado?

**Datos necesarios:** `hecho_contratos` + `dim_geografia`.

**Utilidad:** apoya decisiones de focalización territorial de política
pública y de seguimiento a la ejecución presupuestal regional.


In [ ]:
resultado_5 = spark.sql(f"""
SELECT
    g.departamento,
    COUNT(*) AS numero_contratos,
    ROUND(SUM(h.valor_del_contrato), 2) AS valor_total
FROM {DB_NAME}.hecho_contratos h
LEFT JOIN {DB_NAME}.dim_geografia g
    ON h.geografia_id_geografia = g.id_geografia_dane
GROUP BY g.departamento
ORDER BY valor_total DESC
""")
resultado_5.show(n=15, truncate=False)


### Consulta 6 — JOIN entre hecho y dimensión: top 10 entidades por valor contratado

**Pregunta:** ¿Qué entidades estatales han contratado el mayor valor
acumulado?

**Datos necesarios:** `hecho_contratos` + `dim_entidad`.

**Utilidad:** identifica las entidades con mayor exposición contractual,
útiles como foco de análisis de riesgo o de seguimiento presupuestal.


In [ ]:
resultado_6 = spark.sql(f"""
SELECT
    e.nombre_entidad,
    e.nit_entidad,
    COUNT(*) AS numero_contratos,
    ROUND(SUM(h.valor_del_contrato), 2) AS valor_total
FROM {DB_NAME}.hecho_contratos h
INNER JOIN {DB_NAME}.dim_entidad e
    ON h.entidad_codigo_entidad = e.codigo_entidad
GROUP BY e.nombre_entidad, e.nit_entidad
ORDER BY valor_total DESC
LIMIT 10
""")
resultado_6.show(truncate=False)


### Consulta 7 — JOIN entre hechos y al menos tres dimensiones

**Pregunta:** ¿Cómo se distribuye el valor contratado al cruzar
simultáneamente sector, modalidad de contratación y departamento?

**Datos necesarios:** `hecho_contratos` + `dim_sector` + `dim_modalidad_contratacion` + `dim_geografia`.

**Utilidad:** este tipo de cruce multidimensional es exactamente lo que
justifica construir un cubo en lugar de trabajar con la tabla plana
original — permite responder preguntas de negocio que combinan varias
perspectivas analíticas en una sola consulta.


In [ ]:
resultado_7 = spark.sql(f"""
SELECT
    s.sector,
    m.modalidad_de_contratacion,
    g.departamento,
    COUNT(*) AS numero_contratos,
    ROUND(SUM(h.valor_del_contrato), 2) AS valor_total
FROM {DB_NAME}.hecho_contratos h
LEFT JOIN {DB_NAME}.dim_sector s
    ON h.sector_id_sector = s.id_sector
LEFT JOIN {DB_NAME}.dim_modalidad_contratacion m
    ON h.id_modalidad_de_contratacion = m.id_modalidad_de_contratacion
LEFT JOIN {DB_NAME}.dim_geografia g
    ON h.geografia_id_geografia = g.id_geografia_dane
GROUP BY s.sector, m.modalidad_de_contratacion, g.departamento
ORDER BY valor_total DESC
LIMIT 20
""")
resultado_7.show(truncate=False)


### Consulta 8 — Validación de registros huérfanos

**Pregunta:** ¿Existen contratos cuya llave foránea de entidad o proveedor
no encuentra correspondencia en la dimensión respectiva?

**Datos necesarios:** `hecho_contratos` + `dim_entidad` + `dim_proveedor`.

**Utilidad:** es la validación de integridad referencial del cubo ya
cargado en Hive (complementa la validación hecha en `Transformacion.ipynb`
sobre PLATA, ahora verificada sobre los datos efectivamente cargados en
Hive).


In [ ]:
resultado_8 = spark.sql(f"""
SELECT
    'entidad' AS dimension,
    COUNT(*) AS contratos_huerfanos
FROM {DB_NAME}.hecho_contratos h
LEFT JOIN {DB_NAME}.dim_entidad e
    ON h.entidad_codigo_entidad = e.codigo_entidad
WHERE h.entidad_codigo_entidad IS NOT NULL
  AND e.codigo_entidad IS NULL

UNION ALL

SELECT
    'proveedor' AS dimension,
    COUNT(*) AS contratos_huerfanos
FROM {DB_NAME}.hecho_contratos h
LEFT JOIN {DB_NAME}.dim_proveedor p
    ON h.proveedor_codigo_proveedor = p.codigo_proveedor
WHERE h.proveedor_codigo_proveedor IS NOT NULL
  AND p.codigo_proveedor IS NULL
""")
resultado_8.show(truncate=False)


### Consulta 9 — Análisis temporal: valor contratado por año de firma

**Pregunta:** ¿Cómo ha evolucionado el valor total contratado año a año?

**Datos necesarios:** `hecho_contratos.fecha_de_firma`.

**Utilidad:** permite identificar tendencias de crecimiento o contracción
en la contratación pública y sirve de base para proyecciones. Esta
consulta es la que justifica el particionamiento de `hecho_contratos` por
`anio_firma` en la capa ORO (sección 5): al filtrar/agrupar por año, Spark
puede podar particiones en lugar de leer la tabla completa.


In [ ]:
resultado_9 = spark.sql(f"""
SELECT
    YEAR(fecha_de_firma) AS anio,
    COUNT(*) AS numero_contratos,
    ROUND(SUM(valor_del_contrato), 2) AS valor_total
FROM {DB_NAME}.hecho_contratos
WHERE fecha_de_firma IS NOT NULL
GROUP BY YEAR(fecha_de_firma)
ORDER BY anio
""")
resultado_9.show(n=20, truncate=False)


### Consulta 10 — Pregunta analítica adicional: participación de PYME en el valor contratado

**Pregunta:** ¿Qué proporción del valor total contratado corresponde a
proveedores identificados como PYME (`es_pyme`)?

**Datos necesarios:** `hecho_contratos` + `dim_proveedor`.

**Utilidad:** es un indicador de política pública relevante — el fomento a
la participación de pequeñas y medianas empresas en la contratación
estatal es un objetivo explícito de la normativa colombiana de compras
públicas.


In [ ]:
resultado_10 = spark.sql(f"""
SELECT
    COALESCE(p.es_pyme, 'Sin dato') AS es_pyme,
    COUNT(*) AS numero_contratos,
    ROUND(SUM(h.valor_del_contrato), 2) AS valor_total,
    ROUND(
        100.0 * SUM(h.valor_del_contrato) / SUM(SUM(h.valor_del_contrato)) OVER (), 2
    ) AS porcentaje_del_valor_total
FROM {DB_NAME}.hecho_contratos h
LEFT JOIN {DB_NAME}.dim_proveedor p
    ON h.proveedor_codigo_proveedor = p.codigo_proveedor
GROUP BY COALESCE(p.es_pyme, 'Sin dato')
ORDER BY valor_total DESC
""")
resultado_10.show(truncate=False)


## 9. Análisis general de resultados

Las diez consultas anteriores demuestran que el cubo, tal como quedó
modelado en `CuboDatos.ipynb` y poblado a través de `Extraccion.ipynb` y
`Transformacion.ipynb`, permite responder preguntas de negocio genuinas
sin necesidad de reescribir lógica de limpieza o de relación en cada
consulta: toda esa complejidad ya quedó resuelta en las capas anteriores
del pipeline. El particionamiento por año en `hecho_contratos` beneficia
directamente a la Consulta 9 (y a cualquier consulta futura que filtre por
rango de fechas), mientras que el uso de Parquet en todas las capas reduce
el volumen de I/O en las consultas que solo necesitan un subconjunto de
columnas (por ejemplo, la Consulta 1, que solo cuenta filas).

La validación de huérfanos (Consulta 8) es la comprobación final de que el
proceso de `LEFT JOIN` documentado en `Transformacion.ipynb` no introdujo
inconsistencias entre el hecho y sus dimensiones al momento de cargar en
Hive.


## 10. Conclusiones

- Se persistió el modelo dimensional en la capa ORO, con la tabla de
  hechos particionada por año de firma (decisión justificada por el
  patrón de consultas temporales del taller) y las dimensiones sin
  particionar (por su baja cardinalidad).
- Se cargaron los datos en las tablas Hive creadas en `CuboDatos.ipynb`,
  verificando conteos y esquema tras la carga.
- Se ejecutaron diez consultas SQL que cubren: conteo del hecho,
  agregación de una medida real, análisis por las tres primeras
  dimensiones, un `JOIN` simple, un `JOIN` con al menos tres dimensiones,
  validación de integridad referencial, análisis temporal y una pregunta
  analítica adicional de política pública (participación PYME).
- El cubo de datos queda demostrado como funcional para el análisis de
  contratación pública sobre SECOP II, siguiendo fielmente el modelo
  `CuboContratosPostgreSQL.drawio` a lo largo de los cuatro notebooks del
  taller.
- Los resultados numéricos exactos de cada consulta dependen de los datos
  reales extraídos en `Extraccion.ipynb`; este notebook entrega el código,
  la lógica y la interpretación esperada de cada consulta, pero las cifras
  deben verificarse ejecutando el pipeline completo en el entorno Docker
  del curso.


## 11. Guía para defender el taller

Respuestas de referencia a preguntas que un evaluador podría hacer sobre
las decisiones técnicas tomadas a lo largo de los cuatro notebooks:

1. **¿Por qué PySpark?** Porque permite procesar datos a escala de forma
   distribuida usando una API de alto nivel (DataFrame) en Python,
   evitando escribir MapReduce de bajo nivel, y se integra directamente
   con Hive para persistencia estructurada (Databricks, s.f.).
2. **¿Por qué Spark y no solo Hadoop MapReduce?** Spark mantiene los
   resultados intermedios en memoria en lugar de escribirlos a disco entre
   cada etapa, lo que acelera significativamente los procesos iterativos
   de limpieza y transformación (Chambers y Zaharia, 2018).
3. **¿Por qué un *data lake* en capas (Bronce–Plata–Oro)?** Separa
   claramente el dato crudo (auditable, reprocesable) del dato limpio
   (analítico) y del dato optimizado para consulta, evitando que un error
   de transformación obligue a repetir la extracción completa.
4. **¿Qué significa BRONCE?** Datos tal como llegan de la fuente, sin
   ninguna transformación, usados como respaldo y punto de reproceso.
5. **¿Qué significa PLATA?** Datos limpios, tipados y estructurados en el
   modelo dimensional (hechos y dimensiones), pero aún en Parquet "plano".
6. **¿Qué significa ORO?** Datos finales, optimizados (particionados donde
   se justifica) y cargados en Hive, listos para consulta analítica.
7. **¿Por qué Parquet?** Es un formato columnar que permite leer solo las
   columnas necesarias por consulta, con compresión eficiente, ideal para
   cargas analíticas (OLAP) frente a formatos de fila como CSV o JSON.
8. **¿Por qué Hive?** Provee un catálogo de metadatos y una interfaz SQL
   sobre datos almacenados en el *data lake*, permitiendo consultar el
   cubo con sintaxis SQL estándar sin mover los datos a un motor
   relacional aparte.
9. **¿Cuál es el grano del hecho?** Una fila de `hecho_contratos`
   representa un contrato individual publicado en SECOP II, identificado
   por `id_contrato`.
10. **¿Cuáles son las dimensiones?** Geografía, sector, proveedor,
    entidad, modalidad de contratación, tipo de contrato, producto,
    ordenador de gasto y supervisor.
11. **¿Cómo se relacionan?** A través de llaves foráneas en el hecho que
    apuntan a la llave primaria de cada dimensión (algunas reales de la
    fuente, otras generadas como llave sustituta, documentado en
    `Transformacion.ipynb`).
12. **¿Cómo se trataron los nulos?** Se conservaron como categoría
    explícita cuando representaban un valor de negocio real (p. ej.
    `"No definido"`), y se excluyeron únicamente las filas sin
    `id_contrato`, por no tener grano de hecho válido.
13. **¿Cómo se trataron los duplicados?** Por `id_contrato`, tanto en
    `Extraccion.ipynb` (previo a persistir BRONCE) como al construir la
    tabla de hechos en `Transformacion.ipynb`.
14. **¿Cómo se manejó la paginación?** Con parámetros configurables
    (`LIMIT`, `MAX_RECORDS`) y detención automática al recibir una página
    vacía de la API.
15. **¿Cómo se manejaron errores?** Diferenciando errores de cliente
    (4xx, no reintentables) de errores de servidor o de red (reintentables
    con espera incremental), en `Extraccion.ipynb`.
16. **¿Cómo se validó la calidad?** Con métricas de nulos y duplicados
    antes/después de la limpieza (`Transformacion.ipynb`), y con conteo de
    huérfanos por llave foránea, tanto en PLATA como ya cargado en Hive
    (`Cargue.ipynb`).
17. **¿Qué mejoras se hicieron sobre el ejemplo de referencia?** Manejo
    robusto de errores en la extracción, validación de esquema antes de
    persistir, deduplicación explícita, generación documentada de llaves
    sustitutas, validación de integridad referencial en dos puntos del
    pipeline, y particionamiento justificado por patrón de consulta.
18. **¿Qué preguntas de negocio puede responder el cubo?** Valor
    contratado por sector, modalidad, geografía y entidad; evolución
    temporal del gasto; participación de proveedores PYME; y cualquier
    combinación multidimensional de las anteriores.
19. **¿Por qué el modelo puede responderlas?** Porque separa medidas
    (valores monetarios) de contexto descriptivo (dimensiones), permitiendo
    agregaciones flexibles vía `JOIN` + `GROUP BY` sin reprocesar la fuente
    original.
20. **¿Qué limitaciones tiene la solución?** Cuatro dimensiones dependen
    de llaves sustitutas generadas (no de códigos oficiales, por no estar
    disponibles en la fuente pública consumida); la dimensión de producto
    solo captura el código de categoría principal, sin el desglose UNSPSC
    completo por no estar expuesto por el endpoint; y los volúmenes reales
    procesados dependen del `MAX_RECORDS` configurado en la extracción.


## 12. Lista de evidencias que el estudiante debe capturar

Al ejecutar este pipeline en el entorno Docker del curso, capturar
pantallazo o dejar la salida guardada de:

1. Creación de la `SparkSession` con soporte Hive (cada notebook).
2. `SHOW DATABASES` / `SHOW TABLES` / `DESCRIBE` (`CuboDatos.ipynb`).
3. Log de paginación y conteo total descargado (`Extraccion.ipynb`).
4. Escritura confirmada en BRONCE (`Extraccion.ipynb`).
5. Tabla antes/después de calidad de datos (`Transformacion.ipynb`).
6. Esquema de cada dimensión y del hecho ya construidos
   (`Transformacion.ipynb`).
7. Escritura confirmada en PLATA (`Transformacion.ipynb`).
8. Conteos de tablas cargadas en Hive (`Cargue.ipynb`).
9. Resultado de las diez consultas del cubo (`Cargue.ipynb`).
10. Resultado de la validación de huérfanos ya en Hive (`Cargue.ipynb`).

## 13. Referencias (APA 7)

Chambers, B., & Zaharia, M. (2018). *Spark: The definitive guide*. O'Reilly Media.

Databricks. (s.f.). *What is PySpark?* https://www.databricks.com/glossary/pyspark

Karambelkar, H. V. (2018). *Apache Hadoop 3 quick start guide*. Packt Publishing.
